### TabPFN (minimalist submission)
**Author: [Carl McBride Ellis](https://www.kaggle.com/carlmcbrideellis)** ([LinkedIn](https://www.linkedin.com/in/carl-mcbride-ellis/))

From the paper ["TabPFN: A Transformer That Solves Small Tabular Classification Problems in a Second"](https://arxiv.org/pdf/2207.01848.pdf) by Noah Hollmann, Samuel Müller, Katharina Eggensperger, and Frank Hutter:

> "*TabPFN [is] a single Transformer that has been
pre-trained to approximate probabilistic inference for the novel prior... in a single forward pass, and has thus learned to solve novel small tabular classification tasks (≤ 1 000 training examples, ≤ 100 purely numerical features without missing values and ≤ 10 classes)*"

In [ ]:
import numpy as np
import pandas as pd

### TabPFN
Install [TabPFN](https://github.com/automl/TabPFN) (version 0.1.9) from the [TabPFN (0.1.9) whl](https://www.kaggle.com/datasets/carlmcbrideellis/tabpfn-019-whl)

In [ ]:
!pip install -q /kaggle/input/tabpfn-019-whl/tabpfn-0.1.9-py3-none-any.whl

!mkdir /opt/conda/lib/python3.10/site-packages/tabpfn/models_diff
!cp /kaggle/input/tabpfn-019-whl/prior_diff_real_checkpoint_n_0_epoch_100.cpkt /opt/conda/lib/python3.10/site-packages/tabpfn/models_diff/

In [ ]:
from tabpfn import TabPFNClassifier
# options as per https://www.kaggle.com/code/muelsamu/simple-tabpfn-approach-for-score-of-15-in-1-min by Samuel
classifier = TabPFNClassifier(N_ensemble_configurations=64,device='cuda:0')

### The big three: XGBoost, LightGBM and CatBoost
Each with no hyper-parameter tuning (uncomment to use)

In [ ]:
#from xgboost import XGBClassifier
#classifier = XGBClassifier()

In [ ]:
# from lightgbm import LGBMClassifier
# classifier = LGBMClassifier()

In [ ]:
# from catboost import CatBoostClassifier
# classifier = CatBoostClassifier()

### Data cleaning
From [the TabPFN GitHub page](https://github.com/automl/TabPFN):

> "*Do not preprocess inputs to TabPFN. TabPFN pre-processes inputs internally. It applies a z-score normalization (x-train_x.mean()/train_x.std()) per feature (fitted on the training set) and log-scales outliers heuristically. Finally, TabPFN applies a PowerTransform to all features for every second ensemble member. Pre-processing is important for the TabPFN to make sure that the real-world dataset lies in the distribution of the synthetic datasets seen during training. So to get the best results, do not apply a PowerTransformation to the inputs.*

Thus here we perform the bare minimum of data cleaning and preprocessing

In [ ]:
def cleaning(dataset):
    dataset["EJ"] = dataset["EJ"].replace({"B":0,"A":1})
    dataset = dataset.fillna(dataset.mean(numeric_only=True))
    return dataset

In [ ]:
# adapted from https://www.kaggle.com/code/muelsamu/simple-tabpfn-approach-for-score-of-15-in-1-min
def rebalance(p):
    class_0_est_instances = p[:,0].sum()
    others_est_instances  = p[:,1:].sum()
    new_p = p * np.array([[1/(class_0_est_instances if i==0 else others_est_instances) for i in range(p.shape[1])]])
    p = new_p / np.sum(new_p,axis=1,keepdims=1)
    return p

Read in the training data (617 rows and 56 features)

In [ ]:
dataset = pd.read_csv("/kaggle/input/icr-identify-age-related-conditions/train.csv", index_col="Id")

In [ ]:
X = dataset
X = X.drop(["Class"], axis=1) 
X = cleaning(X)
y = dataset["Class"]

Train the classifier

In [ ]:
%%time

classifier.fit(X, y)

Prediction

In [ ]:
X_test = pd.read_csv("/kaggle/input/icr-identify-age-related-conditions/test.csv", index_col="Id")
X_test = cleaning(X_test)

In [ ]:
%%time

y_pred = classifier.predict_proba(X_test)

In [ ]:
y_pred  = rebalance(y_pred)
class_0 = y_pred[:,0]
class_1 = 1-class_0

Produce a Kaggle submission file

In [ ]:
sample_submission = pd.read_csv("/kaggle/input/icr-identify-age-related-conditions/sample_submission.csv")
sample_submission["class_0"] = class_0
sample_submission["class_1"] = class_1
sample_submission.to_csv('submission.csv',index=False)

### Results
| Classifier | Public LB score | Private LB score
| --- | --- | --- |
| CatBoost | 0.20 | 0.41 |
| TabPFN | 0.20 | 0.48 |
| XGBoost | 0.22 | 0.54 |
| LightGBM | 0.28 | 0.72 |